In [4]:
from pathlib import Path
import geopandas as gpd
import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

TRACK_FILE = Path(
    r"./EP122026_pts.shp"
)


# ============================================================
# READ NHC BEST TRACK
# ============================================================

gdf = gpd.read_file(
    TRACK_FILE
)


print("=" * 90)
print("NHC BEST TRACK")
print("=" * 90)

print("Track points:", len(gdf))

print()
print("Columns:")
print(list(gdf.columns))


# ============================================================
# BUILD DATETIME FROM:
#
# YEAR
# MONTH
# DAY
# HHMM
# ============================================================

year = (
    pd.to_numeric(
        gdf["YEAR"],
        errors="coerce"
    )
    .astype("Int64")
)


month = (
    pd.to_numeric(
        gdf["MONTH"],
        errors="coerce"
    )
    .astype("Int64")
)


day = (
    pd.to_numeric(
        gdf["DAY"],
        errors="coerce"
    )
    .astype("Int64")
)


hhmm = (
    pd.to_numeric(
        gdf["HHMM"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)


# ============================================================
# HHMM -> HOUR + MINUTE
# ============================================================

hour = (
    hhmm // 100
)

minute = (
    hhmm % 100
)


# ============================================================
# CREATE DATETIME
# ============================================================

gdf["track_time"] = pd.to_datetime(
    {
        "year": year,
        "month": month,
        "day": day,
        "hour": hour,
        "minute": minute,
    },
    errors="coerce",
)


# ============================================================
# REMOVE INVALID TIMES
# ============================================================

valid = gdf[
    gdf["track_time"].notna()
].copy()


if valid.empty:

    raise RuntimeError(
        "No valid timestamps could be created."
    )


valid = valid.sort_values(
    "track_time"
).reset_index(
    drop=True
)


# ============================================================
# FIRST / LAST TIMES
# ============================================================

first_time = valid[
    "track_time"
].min()

last_time = valid[
    "track_time"
].max()

first_day = first_time.date()

last_day = last_time.date()


print()
print("=" * 90)
print("TRACK TIME RANGE")
print("=" * 90)

print(
    "First track time:",
    first_time,
    "UTC"
)

print(
    "First track day :",
    first_day
)

print()

print(
    "Last track time :",
    last_time,
    "UTC"
)

print(
    "Last track day  :",
    last_day
)


# ============================================================
# FIRST TRACK POINT
# ============================================================

first_point = valid.iloc[0]


print()
print("=" * 90)
print("FIRST TRACK POINT")
print("=" * 90)

print(
    "Time      :",
    first_point["track_time"],
    "UTC"
)

print(
    "Latitude  :",
    first_point["LAT"]
)

print(
    "Longitude :",
    first_point["LON"]
)

print(
    "Storm type:",
    first_point["STORMTYPE"]
)

print(
    "Intensity :",
    first_point["INTENSITY"]
)

print(
    "SS        :",
    first_point["SS"]
)


# ============================================================
# LAST TRACK POINT
# ============================================================

last_point = valid.iloc[-1]


print()
print("=" * 90)
print("LAST TRACK POINT")
print("=" * 90)

print(
    "Time      :",
    last_point["track_time"],
    "UTC"
)

print(
    "Latitude  :",
    last_point["LAT"]
)

print(
    "Longitude :",
    last_point["LON"]
)

print(
    "Storm type:",
    last_point["STORMTYPE"]
)

print(
    "Intensity :",
    last_point["INTENSITY"]
)

print(
    "SS        :",
    last_point["SS"]
)


# ============================================================
# ADD DATE COLUMN
# ============================================================

valid["date"] = (
    valid["track_time"]
    .dt.date
)


# ============================================================
# ALL POINTS ON FIRST TRACK DAY
# ============================================================

first_day_points = valid[
    valid["date"] == first_day
].copy()


print()
print("=" * 90)
print(
    f"POINTS ON FIRST TRACK DAY: {first_day}"
)
print("=" * 90)


for _, row in first_day_points.iterrows():

    print(
        row["track_time"],
        "|",
        f"Lon {row['LON']:.2f}",
        "|",
        f"Lat {row['LAT']:.2f}",
        "|",
        f"Type {row['STORMTYPE']}",
        "|",
        f"Intensity {row['INTENSITY']}",
        "|",
        f"SS {row['SS']}"
    )


# ============================================================
# ALL POINTS ON LAST TRACK DAY
# ============================================================

last_day_points = valid[
    valid["date"] == last_day
].copy()


print()
print("=" * 90)
print(
    f"POINTS ON LAST TRACK DAY: {last_day}"
)
print("=" * 90)


for _, row in last_day_points.iterrows():

    print(
        row["track_time"],
        "|",
        f"Lon {row['LON']:.2f}",
        "|",
        f"Lat {row['LAT']:.2f}",
        "|",
        f"Type {row['STORMTYPE']}",
        "|",
        f"Intensity {row['INTENSITY']}",
        "|",
        f"SS {row['SS']}"
    )

NHC BEST TRACK
Track points: 58

Columns:
['STORMNAME', 'DTG', 'YEAR', 'MONTH', 'DAY', 'HHMM', 'MSLP', 'BASIN', 'STORMNUM', 'STORMTYPE', 'INTENSITY', 'SS', 'LAT', 'LON', 'geometry']

TRACK TIME RANGE
First track time: 2026-08-27 18:00:00 UTC
First track day : 2026-08-27

Last track time : 2026-09-11 00:00:00 UTC
Last track day  : 2026-09-11

FIRST TRACK POINT
Time      : 2026-08-27 18:00:00 UTC
Latitude  : 11.2
Longitude : -134.0
Storm type: TS
Intensity : 35.0
SS        : 0.0

LAST TRACK POINT
Time      : 2026-09-11 00:00:00 UTC
Latitude  : 29.0
Longitude : -169.5
Storm type: LO
Intensity : 45.0
SS        : 0.0

POINTS ON FIRST TRACK DAY: 2026-08-27
2026-08-27 18:00:00 | Lon -134.00 | Lat 11.20 | Type TS | Intensity 35.0 | SS 0.0

POINTS ON LAST TRACK DAY: 2026-09-11
2026-09-11 00:00:00 | Lon -169.50 | Lat 29.00 | Type LO | Intensity 45.0 | SS 0.0
